# 01 - Lens smoke test  (PLAN.md 5.0, gate V1)

**Hours 1-3.** Load the model and the *pre-fitted* Jacobian lens, and confirm the
readout is sane before anything expensive runs.

Two things to get right here, both cheap to get wrong:

- Use the **`-it` lens**, not the base one. `cfg.lens_id` already encodes this.
- Develop on `debug` (270m-it). Only switch `NANDA_PRESET=target` once the whole
  pipeline runs end to end.

**V1:** lens loads, and a jlens readout on a known prompt gives sensible top tokens.


In [3]:
# --- standard header ------------------------------------------------------
%load_ext autoreload
%autoreload 2

import sys, pathlib
for p in ("/workspace/NandaProj", ".."):
    if p not in sys.path:
        sys.path.insert(0, p)

from nandaproj import config

cfg = config.get_model_config()      # NANDA_PRESET env var, defaults to debug
config.ensure_dirs()
print("preset:", cfg.name, "|", cfg.n_params, "|", cfg.dtype)
print("device:", config.get_device())
print("results ->", config.RESULTS)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
preset: google/gemma-3-270m-it | 270M | bfloat16
device: cuda
results -> /workspace/results


In [2]:
import jlens

In [8]:
def gate(name, ok, detail=""):
    """PLAN.md section 6 verification gate. Fails loudly and stops the notebook."""
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}  {detail}")
    if not ok:
        raise AssertionError(f"gate {name} failed: {detail}")


In [4]:
# Model. bf16 on GPU; the debug model is small enough to be near-instant.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(cfg.name, cache_dir=str(config.HF_CACHE))
model = AutoModelForCausalLM.from_pretrained(
    cfg.name,
    cache_dir=str(config.HF_CACHE),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(model.config.num_hidden_layers, "layers,", model.config.hidden_size, "d_model")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

18 layers, 640 d_model


In [16]:
config.LENS_REPO

'neuronpedia/jacobian-lens'

In [5]:
# Pre-fitted lens from neuronpedia/jacobian-lens -- no fitting required.
from huggingface_hub import snapshot_download

lens_path = snapshot_download(
    repo_id=config.LENS_REPO,
    allow_patterns=[f"{cfg.lens_id}/*"],
    cache_dir=str(config.HF_CACHE),
)
print("lens at", lens_path)

# TODO: load J_l from lens_path. Inspect the repo layout first -- do not assume
# a filename. Record the actual layout here once seen, so 05 can rely on it.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

lens at /workspace/hf_cache/models--neuronpedia--jacobian-lens/snapshots/0731326edff4ae730ffc5356fe1a4728c748b3a6


In [ ]:
# V1: Load and apply Jacobian lens using jlens library (the correct way)
import jlens
import torch

print("Testing Jacobian lens with Jake/Judy prompt")
print("=" * 70)

# Wrap the model with jlens interface
model_jlens = jlens.from_hf(model, tok)
print(f"Wrapped model: {model_jlens.n_layers} layers, d_model={model_jlens.d_model}")

# Load the pre-fitted lens
lens = jlens.JacobianLens.from_pretrained(
    config.LENS_REPO,
    filename=f"{cfg.lens_id}/jlens/Salesforce-wikitext/{cfg.lens_id}_jacobian_lens.pt",
)
print(f"Lens loaded successfully")

# Test prompt
prompt = "Jake and Judy were talking to each other. Jake then handed his toy to"
print(f"\nPrompt: {prompt!r}\n")

# Apply the lens at multiple layers
layers_to_test = [
    model_jlens.n_layers // 4,      # 1/4 way through
    model_jlens.n_layers // 2,      # Middle
    model_jlens.n_layers // 4 * 3,  # 3/4 way through
    model_jlens.n_layers - 2,       # Near end
]

# Get both J-lens and vanilla logit lens predictions
jlens_logits, model_logits, _ = lens.apply(
    model_jlens, prompt, layers=layers_to_test, positions=[-1]
)
logit_lens, _, _ = lens.apply(
    model_jlens, prompt, layers=layers_to_test, positions=[-1], use_jacobian=False
)

print("## J-lens (Jacobian) vs Logit lens predictions:")
print("-" * 70)

def top5(logits):
    """Get top-5 token predictions"""
    return [tok.decode([t]) for t in logits.topk(5).indices]

for layer in layers_to_test:
    print(f"\nLayer {layer:2d}:")
    print(f"  Logit lens:  {top5(logit_lens[layer][0])}")
    print(f"  J-lens:      {top5(jlens_logits[layer][0])}")

print(f"\nFinal layer (model output): {top5(model_logits[0])}")

# Sanity check: verify the lens produces sensible interpretations
print("\n" + "=" * 70)
print("## Sanity Check:")
print("-" * 70)

# At the last token position, we should see sensible next tokens
final_predictions = top5(jlens_logits[layers_to_test[-1]][0])
print(f"J-lens predictions at layer {layers_to_test[-1]}: {final_predictions}")

# Check if we get English tokens (not garbage like before)
all_english = all(
    all(ord(c) < 128 for c in tok_str if c.strip())
    for tok_str in final_predictions
)

print(f"All predictions are ASCII/English-like: {all_english}")

if all_english:
    print("✓ Lens appears to be working correctly!")
    success = True
else:
    print("⚠ Still getting non-English output, may indicate lens-model mismatch")
    success = False

gate("V1 lens readout", success, "Applied J-lens using jlens library")

Testing Jacobian lens with Jake/Judy prompt
Wrapped model: 18 layers, d_model=640


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Lens loaded successfully

Prompt: 'Jake and Judy were talking to each other. Jake then handed his toy to'

## J-lens (Jacobian) vs Logit lens predictions:
----------------------------------------------------------------------

Layer  4:
  Logit lens:  ['ккей', 'setPrototypeOf', 'ప్పటికీ', '../../', 'новні']
  J-lens:      ['oler', ' margarine', 'belly', ' piglets', 'fila']

Layer  9:
  Logit lens:  [' việc', 'гә', 'bersihan', 'yect', 'ضو']
  J-lens:      [' volunteers', ' kids', ' petting', ' inanimate', ' toddlers']

Layer 12:
  Logit lens:  [' Judy', 'Judy', 'JUD', ' JUD', ' Jud']
  J-lens:      [' Judy', 'Judy', ' Judith', ' Julie', ' Janet']

Layer 16:
  Logit lens:  [' Judy', 'Judy', ' them', ' him', ' the']
  J-lens:      [' Judy', 'Judy', ' the', ' them', ' Judith']

Final layer (model output): [' Judy', 'Judy', ' her', ' the', ' a']

## Sanity Check:
----------------------------------------------------------------------
J-lens predictions at layer 16: [' Judy', 'Judy', ' the', 

### Record before moving on

- Which layers have a fitted `J_l`?
- What `k` corresponds to the ~6-10% variance figure? (05 sweeps `k`, but this
  is the anchor point.)
- Wall-clock to load model + lens, so hours 4-8 can be budgeted honestly.
